Do leave out one patient for fig 5

# Header

In [1]:
from utils import *

In [2]:
import sys
sys.path.insert(0, '/n/data1/dfci/surgonc/mittendorf/lab/chelsea/iw/scRNA_v2/codes/packages/cna-master/src')
import cna

In [3]:
import matplotlib.pyplot as plt
import matplotlib as mpl

In [4]:
os.chdir('../')

In [5]:
sc.settings.set_figure_params(dpi=600, dpi_save=600)  # 600 dpi for publication
sc.settings.figdir = "plots/Figure1"

rcParams.update({
    'font.family': 'Arial',
    'svg.fonttype': 'none',    # Keep SVG text as font
    'font.size': 6,            # Main font size (5–7pt recommended)
    'axes.titlesize': 7,
    'axes.labelsize': 6,
    'xtick.labelsize': 5,
    'ytick.labelsize': 5,
    'legend.fontsize': 6,
    'legend.title_fontsize': 6,
    'figure.dpi': 600,

    # --- Nature Immunology-style axes/ticks ---
    'axes.linewidth': 0.5,          # Axis lines: 0.5 pt
    'xtick.major.width': 0.5,        # Major tick lines
    'ytick.major.width': 0.5,
    'xtick.minor.width': 0.5,
    'ytick.minor.width': 0.5,
    'lines.linewidth': 0.7,          # Plot/data lines
    'axes.grid': False,              # No grid lines

    # --- Ticks on all sides ---
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.top': False,
    'xtick.bottom': False,
    'ytick.left': True,
    'ytick.right': False,


    # --- Tick label padding (minimal) ---
    'xtick.major.pad': 0,
    'ytick.major.pad': 0,

    # ---- Thin lines for Nature Immunology style ----
    'lines.linewidth': 0.5,     # General line width (default: 1.5)
    'patch.linewidth': 0.5,     # Outline for violin/bars (default: 1.0)
    
})

# plt.rcParams['axes.grid'] = False      # Extra guarantee, if using seaborn
# sns.set_style("white")                 # Removes upper/right axes and grid

# -- Scanpy vector-friendly mode --
sc.settings._vector_friendly = True

In [6]:
with open('data/Legends/celltypelevel3_palette.pkl','rb') as f:
    palette_all = pkl.load(f)

palette_all

{'Naive B': (0.843137, 0.0, 0.0),
 'Mature B': (0.54902, 0.235294, 1.0),
 'Memory B': (0.007843, 0.533333, 0.0),
 'Plasmablast': (0.0, 0.67451, 0.780392),
 'Naive CD4T': (0.596078, 1.0, 0.0),
 'Central Memory CD4T': (1.0, 0.498039, 0.819608),
 'Effector Memory CD4T': (0.423529, 0.0, 0.309804),
 'Cytotoxic CD4T': (1.0, 0.647059, 0.188235),
 'Treg': (0.345098, 0.231373, 0.0),
 'Naive CD8T': (0.0, 0.341176, 0.34902),
 'Central Memory CD8T': (0.0, 0.0, 0.866667),
 'Effector Memory CD8T': (0.0, 0.992157, 0.811765),
 'Double Negative T': (0.631373, 0.458824, 0.415686),
 'γδ T': (0.737255, 0.717647, 1.0),
 'MAIT': (0.584314, 0.709804, 0.470588),
 'cDC1': (0.752941, 0.015686, 0.72549),
 'cDC2': (0.392157, 0.329412, 0.454902),
 'pDC': (0.47451, 0.0, 0.0),
 'Classical Monocyte': (0.027451, 0.454902, 0.847059),
 'Intermediate Monocyte': (0.996078, 0.960784, 0.564706),
 'Nonclassical Monocyte': (0.0, 0.294118, 0.0),
 'Cytotoxic Monocyte': (0.560784, 0.478431, 0.0),
 'CD56-dim NK': (1.0, 0.447059, 

In [7]:
# major_celltype_palette = {
#     'B':    '#3072C4',   # strong blue
#     'T':    '#EF763F',   # bold orange
#     'DC':   '#35A293',   # teal
#     'Mono': '#B3813E',   # rich brown/gold
#     'NK':   '#7D4C99',   # deep purple
# }

cells_group = {
    'B':['Naive B',
 'Mature B',
 'Memory B',
 'Plasmablast'],
    'T':['Naive CD4T',
 'Central Memory CD4T',
 'Effector Memory CD4T',
 'Cytotoxic CD4T',
 'Treg',
 'Naive CD8T',
 'Central Memory CD8T',
 'Effector Memory CD8T',
 'Double Negative T',
 'γδ T',
 'MAIT'],
    'DC':[
   'cDC1',
 'cDC2',
 'pDC'     
    ],
    'Mono':[
        'Classical Monocyte',
 'Intermediate Monocyte',
 'Nonclassical Monocyte',
 'Cytotoxic Monocyte'
        ],
    'NK':[
        'CD56-dim NK',
 'CD56-bright NK',
 'Proliferating NK'
    ]
}

# Prep data

In [8]:
meta = pd.read_csv('data/01_meta/meta_pt.csv')#, index_col=0)
#meta.index = [x.replace('_','-') for x in meta.index]

meta['race-stage'] = meta['race'].astype(str) + '-' + meta['group'].astype(str)
meta = meta.drop(columns = ['race_stage'])

meta['race-stage'] = pd.Categorical(
    meta['race-stage'],
    categories=['NHW-HRL', 'NHW-DCIS', 'NHW-BC','NHB-HRL', 'NHB-DCIS', 'NHB-BC'],
    ordered=True
    )

meta

,sampleid,dataset_name,batch,viability_pct,group,race,age,diabetes,obesity,race-stage
0,9-D,Pool137_7,1,95,DCIS,NHW,37.0,N,Y,NHW-DCIS
1,30-D,Pool137_12,1,92,DCIS,NHW,78.0,N,N,NHW-DCIS
2,36-D,Pool137_18,1,93,DCIS,NHB,39.0,N,Y,NHB-DCIS
3,119-D,Pool137_20,1,95,DCIS,NHB,80.0,N,Y,NHB-DCIS
4,28-D,Pool137_24,1,94,DCIS,NHW,53.0,N,N,NHW-DCIS
5,53-D,Pool137_27,1,96,DCIS,NHB,45.0,N,N,NHB-DCIS
6,77-D,Pool137_30,1,90,DCIS,NHW,45.0,N,N,NHW-DCIS
7,31-D,Pool138_6,2,93,DCIS,NHW,44.0,N,N,NHW-DCIS
8,6-H,Pool137_6,1,94,HRL,NHW,71.1,N,Y,NHW-HRL
9,1-H,Pool137_10,1,88,HRL,NHW,62.3,N,N,NHW-HRL


In [9]:
dat = sc.read_h5ad(
    'data/03_cleandata/ft_all.h5ad'
)

dat

AnnData object with n_obs × n_vars = 434565 × 27879
    obs: 'nFeaturess_RNA', 'nFeaturess_RNA_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublets_score', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'predicted.celltype.l1.score', 'predicted.celltype.l1', 'predicted.celltype.l2.score', 'predicted.celltype.l2', 'predicted.celltype.l3.score', 'predicted.celltype.l3', 'mapping.score', 'RNA_snn_res.0.8', 'seurat_clusters', 'dataset_name', 'RNA_snn_res.0.1', 'S.Score', 'G2M.Score', 'Phase', 'celltype_level3'
    var: 'vf_vst_counts.Pool137_10_mean', 'vf_vst_counts.Pool137_10_variance', 'vf_vst_counts.Pool137_10_variance.expected', 'vf_vst_counts.Pool137_10_variance.standardized', 'vf_vst_counts.Pool137_10_variable', 'vf_vst_counts.Pool137_10_rank', 'vf_vst_counts.Pool137_11_mean', 'vf_vst_counts.Pool137_11_variance', 'vf_vst_counts.Pool137_11_variance.expected', 'vf_vst_counts.Pool137_11_variance.standardized', 'vf_vst_counts.Pool137_11_variable', 'vf_vst_counts.Pool137

In [10]:
for col in ['sampleid', 'age', 'race-stage', 'race', 'group', 'diabetes', 'obesity']:
    dat.obs[col] = dat.obs['dataset_name'].map(meta.set_index('dataset_name')[col])
    
for xcol in ['diabetes','obesity']:
    dat.obs[xcol] = [1 if x=='Y' else 0 for x in dat.obs[xcol]]

xcol = 'group'
dat.obs[xcol] = [0 if x=='HRL' else 1 if x=='DCIS' else 2 for x in dat.obs[xcol]]

xcol = 'race'
dat.obs[xcol] = [0 if x=='NHW' else 1 if x=='NHB' else None for x in dat.obs[xcol]]

dat.obs.columns

Index(['nFeaturess_RNA', 'nFeaturess_RNA_by_counts', 'total_counts',
       'total_counts_mt', 'pct_counts_mt', 'doublets_score', 'nCount_RNA',
       'nFeature_RNA', 'percent.mt', 'predicted.celltype.l1.score',
       'predicted.celltype.l1', 'predicted.celltype.l2.score',
       'predicted.celltype.l2', 'predicted.celltype.l3.score',
       'predicted.celltype.l3', 'mapping.score', 'RNA_snn_res.0.8',
       'seurat_clusters', 'dataset_name', 'RNA_snn_res.0.1', 'S.Score',
       'G2M.Score', 'Phase', 'celltype_level3', 'sampleid', 'age',
       'race-stage', 'race', 'group', 'diabetes', 'obesity'],
      dtype='object')

In [11]:
dat.obs.head(3)

,nFeaturess_RNA,nFeaturess_RNA_by_counts,total_counts,total_counts_mt,pct_counts_mt,doublets_score,nCount_RNA,nFeature_RNA,percent.mt,predicted.celltype.l1.score,...,G2M.Score,Phase,celltype_level3,sampleid,age,race-stage,race,group,diabetes,obesity
Pool137_10_AAACCTGAGACTCGGA-1,1095.0,1095,2669.0,80.0,2.997377,0.039230,2669.0,1095.0,2.997377,0.987477,...,-0.030749,G1,Naive CD4T,1-H,62.3,NHW-HRL,0,0,0,0
Pool137_10_AAACCTGAGATCTGCT-1,2474.0,2472,6188.0,178.0,2.876535,0.021953,6188.0,2472.0,2.876535,1.000000,...,0.004461,S,Nonclassical Monocyte,1-H,62.3,NHW-HRL,0,0,0,0
Pool137_10_AAACCTGAGATTACCC-1,1436.0,1436,3504.0,100.0,2.853881,0.039230,3504.0,1436.0,2.853881,0.996618,...,-0.066766,S,Naive CD8T,1-H,62.3,NHW-HRL,0,0,0,0


# Analysis

In [12]:
import numpy as np
import pandas as pd
import scanpy as sc
from datetime import datetime
import time
import gc

results = []

celltypes_major = [
    'B', 'CD4 T', 'CD8 T', 'other T',
    'DC', 'Mono', 'NK'
]

patients_loo = dat.obs['dataset_name'].dropna().unique()

n_total = len(celltypes_major) * len(patients_loo)
counter = 0

def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


for celltype_major in celltypes_major:

    for pt_loo in patients_loo:

        counter += 1
        t0 = time.time()

        print(
            f"\n[{now()}] Start {counter}/{n_total}: "
            f"celltype={celltype_major}, leave_out={pt_loo}",
            flush=True
        )

        try:
            d = dat[
                (dat.obs['predicted.celltype.l1'] == celltype_major) &
                (dat.obs['dataset_name'] != pt_loo)
            ].copy()

            total_cells_before = d.n_obs

            if d.n_obs < 50:
                print(
                    f"[{now()}] SKIP: too few cells after filtering: {d.n_obs}",
                    flush=True
                )

                results.append({
                    'celltype_major': celltype_major,
                    'pt_loo': pt_loo,
                    'testvar': 'race',
                    'status': 'skipped_too_few_cells',
                    'global_p': np.nan,
                    'totalCells': d.n_obs,
                    'nan%': np.nan,
                    'pos%': np.nan,
                    'neg%': np.nan,
                    'zero%': np.nan,
                    'elapsed_min': (time.time() - t0) / 60
                })

                del d
                gc.collect()
                continue

            print(f"[{now()}] Running neighbors: n_cells={d.n_obs}", flush=True)
            sc.pp.neighbors(d)

            print(f"[{now()}] Building sample metadata", flush=True)

            samplem = cna.ut.obs_to_sample(
                d,
                ['group', 'race', 'age', 'obesity', 'diabetes'],
                'dataset_name'
            )

            samplem['batch'] = [
                'Pool137' if str(x).startswith('Pool137') else 'Pool138'
                for x in samplem.index
            ]

            testvar = 'race'
            covars = ['group', 'age']

            # Add optional covariates only if variable exists and has >1 value after LOO
            for v in ['obesity', 'diabetes']:
                if v in samplem.columns and samplem[v].dropna().nunique() > 1:
                    covars.append(v)

            # Sanity checks after leave-one-out
            if samplem[testvar].dropna().nunique() < 2:
                print(
                    f"[{now()}] SKIP: {testvar} has <2 levels after leaving out {pt_loo}",
                    flush=True
                )

                results.append({
                    'celltype_major': celltype_major,
                    'pt_loo': pt_loo,
                    'testvar': testvar,
                    'status': 'skipped_testvar_one_level',
                    'global_p': np.nan,
                    'totalCells': d.n_obs,
                    'nan%': np.nan,
                    'pos%': np.nan,
                    'neg%': np.nan,
                    'zero%': np.nan,
                    'elapsed_min': (time.time() - t0) / 60
                })

                del d
                gc.collect()
                continue

            # Drop covariates that became constant after LOO
            covars = [
                v for v in covars
                if v in samplem.columns and samplem[v].dropna().nunique() > 1
            ]

            print(
                f"[{now()}] Running CNA association: "
                f"testvar={testvar}, covars={covars}, n_samples={samplem.shape[0]}",
                flush=True
            )

            p = cna.tl.association(
                d,
                samplem[testvar],
                'dataset_name',
                covs=samplem[covars],
                batches=samplem['batch'],
                key_added=f'{testvar}_coef'
            )

            d.uns[f'cnaGlobalPval_{testvar}'] = p

            coef_col = f"{testvar}_coef"
            fdr_col = f"{testvar}_coef_fdr"
            color_col = f"{testvar}_coef_plot"

            if coef_col not in d.obs.columns or fdr_col not in d.obs.columns:
                raise ValueError(
                    f"Expected CNA columns not found. "
                    f"Missing? coef_col={coef_col in d.obs.columns}, "
                    f"fdr_col={fdr_col in d.obs.columns}"
                )

            d.obs[color_col] = np.where(
                d.obs[fdr_col] < 0.1,
                d.obs[coef_col],
                np.nan
            )

            coln = d.obs[color_col]
            total = len(coln)

            n_nan = coln.isna().sum()
            n_pos = ((~coln.isna()) & (coln > 0)).sum()
            n_neg = ((~coln.isna()) & (coln < 0)).sum()
            n_zero = ((~coln.isna()) & (coln == 0)).sum()

            elapsed_min = (time.time() - t0) / 60

            print(
                f"[{now()}] DONE {counter}/{n_total}: "
                f"celltype={celltype_major}, leave_out={pt_loo}, "
                f"global_p={p:.3g}, elapsed={elapsed_min:.2f} min",
                flush=True
            )

            print(
                f"    total={total}, sanity={n_nan + n_pos + n_neg + n_zero}, "
                f"nan={n_nan}, >0={n_pos}, <0={n_neg}, =0={n_zero}",
                flush=True
            )

            results.append({
                'celltype_major': celltype_major,
                'pt_loo': pt_loo,
                'testvar': testvar,
                'covars': ','.join(covars),
                'status': 'ok',
                'global_p': p,
                'totalCells': total,
                'nan%': n_nan / total * 100,
                'pos%': n_pos / total * 100,
                'neg%': n_neg / total * 100,
                'zero%': n_zero / total * 100,
                'elapsed_min': elapsed_min
            })

            del d
            gc.collect()

        except Exception as e:

            elapsed_min = (time.time() - t0) / 60

            print(
                f"[{now()}] ERROR {counter}/{n_total}: "
                f"celltype={celltype_major}, leave_out={pt_loo}, "
                f"elapsed={elapsed_min:.2f} min",
                flush=True
            )
            print(f"    Error: {repr(e)}", flush=True)

            results.append({
                'celltype_major': celltype_major,
                'pt_loo': pt_loo,
                'testvar': 'race',
                'status': 'error',
                'error': repr(e),
                'global_p': np.nan,
                'totalCells': np.nan,
                'nan%': np.nan,
                'pos%': np.nan,
                'neg%': np.nan,
                'zero%': np.nan,
                'elapsed_min': elapsed_min
            })

            try:
                del d
            except NameError:
                pass

            gc.collect()


df_summary = pd.DataFrame(results)
df_summary.to_csv('data/10_cna/LLO_stats.csv')


[2026-06-24 08:32:02] Start 1/385: celltype=B, leave_out=Pool137_10
[2026-06-24 08:32:09] Running neighbors: n_cells=48933
[2026-06-24 08:32:36] Building sample metadata
[2026-06-24 08:32:36] Running CNA association: testvar=race, covars=['group', 'age', 'obesity', 'diabetes'], n_samples=54


/n/data1/dfci/surgonc/mittendorf/lab/chelsea/iw/scRNA_v2/codes/packages/cna-master/src/cna/utils/multisample.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  d.obs.groupby(by=sid_name)[columns].aggregate(aggregate)
/n/data1/dfci/surgonc/mittendorf/lab/chelsea/iw/scRNA_v2/codes/packages/cna-master/src/cna/tools/_association.py:66: UserWarning: data supported use of 4 NAM PCs, which is the maximum considered. Consider allowing more PCs by using the "ks" argument.
  warnings.warn(('data supported use of {} NAM PCs, which is the maximum considered. '+\
/n/data1/dfci/surgonc/mittendorf/lab/chelsea/iw/scRNA_v2/codes/packages/cna-master/src/cna/tools/_association.py:87: UserWarning: global association p-value attained minimal possible value. Consider increasing Nnull
  warnings.warn('global associati

[2026-06-24 08:32:48] DONE 1/385: celltype=B, leave_out=Pool137_10, global_p=0.000999, elapsed=0.77 min
    total=48933, sanity=48933, nan=46513, >0=34, <0=2386, =0=0

[2026-06-24 08:32:48] Start 2/385: celltype=B, leave_out=Pool137_11
[2026-06-24 08:32:54] Running neighbors: n_cells=48732
[2026-06-24 08:32:59] Building sample metadata
[2026-06-24 08:32:59] Running CNA association: testvar=race, covars=['group', 'age', 'obesity', 'diabetes'], n_samples=54


/n/data1/dfci/surgonc/mittendorf/lab/chelsea/iw/scRNA_v2/codes/packages/cna-master/src/cna/utils/multisample.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  d.obs.groupby(by=sid_name)[columns].aggregate(aggregate)
/n/data1/dfci/surgonc/mittendorf/lab/chelsea/iw/scRNA_v2/codes/packages/cna-master/src/cna/tools/_association.py:66: UserWarning: data supported use of 4 NAM PCs, which is the maximum considered. Consider allowing more PCs by using the "ks" argument.
  warnings.warn(('data supported use of {} NAM PCs, which is the maximum considered. '+\
/n/data1/dfci/surgonc/mittendorf/lab/chelsea/iw/scRNA_v2/codes/packages/cna-master/src/cna/tools/_association.py:87: UserWarning: global association p-value attained minimal possible value. Consider increasing Nnull
  warnings.warn('global associati

[2026-06-24 08:33:11] DONE 2/385: celltype=B, leave_out=Pool137_11, global_p=0.000999, elapsed=0.37 min
    total=48732, sanity=48732, nan=46661, >0=34, <0=2037, =0=0

[2026-06-24 08:33:11] Start 3/385: celltype=B, leave_out=Pool137_12
[2026-06-24 08:33:16] Running neighbors: n_cells=48943
[2026-06-24 08:33:21] Building sample metadata
[2026-06-24 08:33:21] Running CNA association: testvar=race, covars=['group', 'age', 'obesity', 'diabetes'], n_samples=54


...

[2026-06-24 11:03:42] DONE 383/385: celltype=NK, leave_out=Pool138_7, global_p=0.299, elapsed=0.24 min
    total=35782, sanity=35782, nan=35782, >0=0, <0=0, =0=0

[2026-06-24 11:03:42] Start 384/385: celltype=NK, leave_out=Pool138_8
[2026-06-24 11:03:45] Running neighbors: n_cells=36370
[2026-06-24 11:03:48] Building sample metadata
[2026-06-24 11:03:48] Running CNA association: testvar=race, covars=['group', 'age', 'obesity', 'diabetes'], n_samples=54


/n/data1/dfci/surgonc/mittendorf/lab/chelsea/iw/scRNA_v2/codes/packages/cna-master/src/cna/utils/multisample.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  d.obs.groupby(by=sid_name)[columns].aggregate(aggregate)


[2026-06-24 11:03:56] DONE 384/385: celltype=NK, leave_out=Pool138_8, global_p=0.427, elapsed=0.24 min
    total=36370, sanity=36370, nan=36370, >0=0, <0=0, =0=0

[2026-06-24 11:03:57] Start 385/385: celltype=NK, leave_out=Pool138_9
[2026-06-24 11:03:59] Running neighbors: n_cells=36143
[2026-06-24 11:04:03] Building sample metadata
[2026-06-24 11:04:03] Running CNA association: testvar=race, covars=['group', 'age', 'obesity', 'diabetes'], n_samples=54


/n/data1/dfci/surgonc/mittendorf/lab/chelsea/iw/scRNA_v2/codes/packages/cna-master/src/cna/utils/multisample.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  d.obs.groupby(by=sid_name)[columns].aggregate(aggregate)


[2026-06-24 11:04:11] DONE 385/385: celltype=NK, leave_out=Pool138_9, global_p=0.499, elapsed=0.24 min
    total=36143, sanity=36143, nan=36143, >0=0, <0=0, =0=0


In [13]:
df_summary

,celltype_major,pt_loo,testvar,covars,status,global_p,totalCells,nan%,pos%,neg%,zero%,elapsed_min
0,B,Pool137_10,race,"group,age,obesity,diabetes",ok,0.000999,48933,95.054462,0.069483,4.876055,0.0,0.771649
1,B,Pool137_11,race,"group,age,obesity,diabetes",ok,0.000999,48732,95.750226,0.069769,4.180005,0.0,0.372564
2,B,Pool137_12,race,"group,age,obesity,diabetes",ok,0.000999,48943,94.383262,0.181844,5.434894,0.0,0.369978
3,B,Pool137_14,race,"group,age,obesity,diabetes",ok,0.000999,48053,93.763137,0.149835,6.087029,0.0,0.356233
4,B,Pool137_15,race,"group,age,obesity,diabetes",ok,0.000999,48488,95.751526,0.049497,4.198977,0.0,0.348831
...,...,...,...,...,...,...,...,...,...,...,...,...
380,NK,Pool138_5,race,"group,age,obesity,diabetes",ok,0.382617,35288,100.000000,0.000000,0.000000,0.0,0.231721
381,NK,Pool138_6,race,"group,age,obesity,diabetes",ok,0.375624,36535,100.000000,0.000000,0.000000,0.0,0.237158
382,NK,Pool138_7,race,"group,age,obesity,diabetes",ok,0.298701,35782,100.000000,0.000000,0.000000,0.0,0.236237
383,NK,Pool138_8,race,"group,age,obesity,diabetes",ok,0.426573,36370,100.000000,0.000000,0.000000,0.0,0.236201


In [18]:
import pandas as pd
import numpy as np

def median_range(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    return f"{x.median():.3g} ({x.min():.3g}-{x.max():.3g})"

def n_pct_condition(mask):
    mask = mask.dropna()
    n = mask.sum()
    denom = len(mask)
    pct = n / denom * 100 if denom > 0 else np.nan
    return f"{n}/{denom} ({pct:.1f}%)"

summary_by_celltype = (
    df_summary
    .groupby("celltype_major")
    .agg(
        global_p_median_range=("global_p", median_range),
        global_p_lt_0_05_n_pct=(
            "global_p",
            lambda x: n_pct_condition(pd.to_numeric(x, errors="coerce") < 0.05)
        ),
        pos_gt0_n_pct=(
            "pos%",
            lambda x: n_pct_condition(pd.to_numeric(x, errors="coerce") > 0)
        ),
        neg_gt0_n_pct=(
            "neg%",
            lambda x: n_pct_condition(pd.to_numeric(x, errors="coerce") > 0)
        ),
    )
    .reset_index()
)

combined = (
    df_summary
    .groupby("celltype_major")
    .apply(
        lambda d: n_pct_condition(
            (pd.to_numeric(d["global_p"], errors="coerce") < 0.05)
            & (pd.to_numeric(d["pos%"], errors="coerce") > 0)
            & (pd.to_numeric(d["neg%"], errors="coerce") > 0)
        )
    )
    .reset_index(name="global_p_lt_0_05_pos_gt0_neg_gt0_n_pct")
)

summary_by_celltype = summary_by_celltype.merge(combined, on="celltype_major")

summary_by_celltype

/tmp/ipykernel_3770805/3198807504.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,celltype_major,global_p_median_range,global_p_lt_0_05_n_pct,pos_gt0_n_pct,neg_gt0_n_pct,global_p_lt_0_05_pos_gt0_neg_gt0_n_pct
0,B,0.000999 (0.000999-0.005),55/55 (100.0%),51/55 (92.7%),53/55 (96.4%),51/55 (92.7%)
1,CD4 T,0.00999 (0.002-0.0519),54/55 (98.2%),33/55 (60.0%),34/55 (61.8%),32/55 (58.2%)
2,CD8 T,0.135 (0.0509-0.319),0/55 (0.0%),0/55 (0.0%),0/55 (0.0%),0/55 (0.0%)
3,DC,0.003 (0.000999-0.00899),55/55 (100.0%),53/55 (96.4%),53/55 (96.4%),53/55 (96.4%)
4,Mono,0.000999 (0.000999-0.000999),55/55 (100.0%),55/55 (100.0%),55/55 (100.0%),55/55 (100.0%)
5,NK,0.352 (0.169-0.629),0/55 (0.0%),0/55 (0.0%),1/55 (1.8%),0/55 (0.0%)
6,other T,0.67 (0.345-0.807),0/55 (0.0%),0/55 (0.0%),0/55 (0.0%),0/55 (0.0%)


In [20]:
dft = summary_by_celltype.iloc[:,:5]

dft.columns = [
   'Cell lineage',
   'Global association P value, median (range)',
   'Global association P <0.05, n (%) LOO runs',
   'Positive local association cell state% >0, n (%) LOO runs',
   'Negative local association cell state% >0, n (%) LOO runs'
   ]

dft

,Cell lineage,"Global association P value, median (range)","Global association P <0.05, n (%) LOO runs","Positive local association cell state% >0, n (%) LOO runs","Negative local association cell state% >0, n (%) LOO runs"
0,B,0.000999 (0.000999-0.005),55/55 (100.0%),51/55 (92.7%),53/55 (96.4%)
1,CD4 T,0.00999 (0.002-0.0519),54/55 (98.2%),33/55 (60.0%),34/55 (61.8%)
2,CD8 T,0.135 (0.0509-0.319),0/55 (0.0%),0/55 (0.0%),0/55 (0.0%)
3,DC,0.003 (0.000999-0.00899),55/55 (100.0%),53/55 (96.4%),53/55 (96.4%)
4,Mono,0.000999 (0.000999-0.000999),55/55 (100.0%),55/55 (100.0%),55/55 (100.0%)
5,NK,0.352 (0.169-0.629),0/55 (0.0%),0/55 (0.0%),1/55 (1.8%)
6,other T,0.67 (0.345-0.807),0/55 (0.0%),0/55 (0.0%),0/55 (0.0%)


In [22]:
dft.to_csv(
    'data/Tables/supp_loo.csv'
)